<a href="https://colab.research.google.com/github/LGLV-Ciencia-de-Datos/Curso_python_Ciencia_de_Datos/blob/main/c)_Exercise_Categorical_Variables.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

¡Codificando **variables categóricas**, obtendrás los mejores resultados hasta ahora!

### **Configuración**

Las preguntas a continuación te darán retroalimentación sobre tu trabajo. Ejecuta la siguiente celda para configurar el sistema de retroalimentación.

In [ ]:
# Carga las librerias necesarias
import pandas as pd
import os
import kagglehub
from sklearn.model_selection import train_test_split

In [ ]:
# Lee los datos
X = pd.read_csv('/kaggle/input/iowa-house-prices/train.csv') # /kaggle/input/home-data-for-ml-course/train.csv
X_test = pd.read_csv('/kaggle/input/iowa-house-prices/test.csv') # /kaggle/input/home-data-for-ml-course/test.csv

# Elimina filas con valores faltantes en el objetivo, separa el objetivo de los predictores.
X.dropna(axis=0, subset=['SalePrice'], inplace=True)
y = X.SalePrice
X.drop(['SalePrice'], axis=1, inplace=True)

# Para simplificar las cosas, eliminaremos las columnas con valores faltantes.
cols_with_missing = [col for col in X.columns if X[col].isnull().any()]
X.drop(cols_with_missing, axis=1, inplace=True)
X_test.drop(cols_with_missing, axis=1, inplace=True)

# Separar el conjunto de validación de los datos de entrenamiento
X_train, X_valid, y_train, y_valid = train_test_split(X, y,
                                                      train_size=0.8, test_size=0.2,
                                                      random_state=0)

Utiliza la siguiente celda de código para imprimir las primeras cinco filas de los datos.

In [ ]:
X_train.head()

Observa que el conjunto de datos contiene tanto variables numéricas como categóricas. Deberás codificar los datos categóricos antes de entrenar un modelo.

Para comparar diferentes modelos, utilizarás la misma función `score_dataset()` del tutorial. Esta función informa del [error absoluto medio](https://en.wikipedia.org/wiki/Mean_absolute_error) (MAE) de un modelo de bosque aleatorio.

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

# function for comparing different approaches
def score_dataset(X_train, X_valid, y_train, y_valid):
    model = RandomForestRegressor(n_estimators=100, random_state=0)
    model.fit(X_train, y_train)
    preds = model.predict(X_valid)
    return mean_absolute_error(y_valid, preds)

### **Paso 1:** Eliminar columnas con datos categóricos

Comenzarás con el enfoque más sencillo. Utiliza la celda de código a continuación para preprocesar los datos en `X_train` y `X_valid`, eliminando las columnas con datos categóricos. Asigna los DataFrames preprocesados a `drop_X_train` y `drop_X_valid`, respectivamente.  

In [ ]:
# Rellena las líneas a continuación: elimina columnas en los datos de entrenamiento y validación
drop_X_train = ____#X_train.select_dtypes(exclude=['object'])
drop_X_valid = ____#X_valid.select_dtypes(exclude=['object'])

Ejecuta la siguiente celda de código para obtener el MAE de este enfoque.

In [ ]:
print("MAE from Approach 1 (Drop categorical variables):")
print(score_dataset(drop_X_train, drop_X_valid, y_train, y_valid))

Antes de pasar a la codificación ordinal, investigaremos el conjunto de datos. Específicamente, analizaremos la columna `'Condition2'`. La celda de código a continuación muestra las entradas únicas tanto en los conjuntos de entrenamiento como en los de validación.

In [ ]:
print("Los valores únicos en la columna 'Condition2' en los datos de entrenamiento:", X_train['Condition2'].unique())
print("\nLos valores únicos en la columna 'Condition2' en los datos de validación:", X_valid['Condition2'].unique())

### **Paso 2:** Codificación ordinal

### Parte A

Si ahora escribes código para:
- ajustar un codificador ordinal a los datos de entrenamiento, y luego
- usarlo para transformar tanto los datos de entrenamiento como los de validación,

obtendrás un error. ¿Puedes ver por qué sucede esto? (_Necesitarás usar la salida anterior para responder a esta pregunta._)

Codificar con un codificador ordinal una columna en los datos de entrenamiento crea una etiqueta numérica correspondiente para cada valor único que aparece en los datos de entrenamiento. En el caso de que los datos de validación contengan valores que no aparecen también en los datos de entrenamiento, el codificador generará un error, porque esos valores no tendrán un entero asignado. Observa que la columna `'Condition2'` en los datos de validación contiene los valores `'RRAn'` y `'RRNn'`, pero estos no aparecen en los datos de entrenamiento; por lo tanto, si intentamos usar un codificador ordinal con `scikit-learn`, el código generará un error.

¿Hay algún valor que aparezca en los datos de validación pero no en los datos de entrenamiento?

Este es un problema común con el que te encontrarás en datos del mundo real, y existen muchas formas de solucionar este problema. Por ejemplo, puedes crear un codificador ordinal personalizado para gestionar nuevas categorías. Sin embargo, la opción más sencilla es eliminar las columnas categóricas problemáticas.

Ejecuta la celda de código a continuación para guardar las columnas problemáticas en una lista de Python llamada `bad_label_cols`. Asimismo, las columnas que pueden codificarse de forma ordinal sin riesgo se almacenan en `good_label_cols`.

In [ ]:
# Columnas categóricas en los datos de entrenamiento
object_cols = [col for col in X_train.columns if X_train[col].dtype == "object"]

# Columnas que pueden ser codificadas ordinalmente sin riesgo
good_label_cols = [col for col in object_cols if
                   set(X_valid[col]).issubset(set(X_train[col]))]

# Columnas problemáticas que serán eliminadas del conjunto de datos
bad_label_cols = list(set(object_cols)-set(good_label_cols))

print('Columnas categóricas que se codificarán ordinalmente:', good_label_cols)
print('\nColumnas categóricas que se eliminarán del conjunto de datos:', bad_label_cols)

### **Parte B**

Utiliza la siguiente celda de código para codificar ordinalmente los datos en `X_train` y `X_valid`. Asigna los DataFrames preprocesados a `label_X_train` y `label_X_valid`, respectivamente.
- A continuación, se proporciona código para eliminar las columnas categóricas en `bad_label_cols` del conjunto de datos.
- Debes codificar ordinalmente las columnas categóricas en `good_label_cols`.  

In [ ]:
from sklearn.preprocessing import OrdinalEncoder

# Drop categorical columns that will not be encoded
label_X_train = X_train.drop(bad_label_cols, axis=1)
label_X_valid = X_valid.drop(bad_label_cols, axis=1)

# Apply ordinal encoder
 # Your code here

ordinal_encoder = ____#OrdinalEncoder()
label_X_train[good_label_cols] = ____#ordinal_encoder.fit_transform(X_train[good_label_cols])
label_X_valid[good_label_cols] = ____#ordinal_encoder.transform(X_valid[good_label_cols])

Ejecute la siguiente celda de código para obtener el MAE de este método.

In [ ]:
print("MAE from Approach 2 (Ordinal Encoding):")
print(score_dataset(label_X_train, label_X_valid, y_train, y_valid))

Hasta ahora, has probado dos enfoques diferentes para tratar las variables categóricas. Y, has visto que codificar los datos categóricos ofrece mejores resultados que eliminar columnas del conjunto de datos.

Pronto, probarás la codificación one-hot. Antes de ello, hay un tema adicional que debemos abordar. Comienza ejecutando la siguiente celda de código sin cambios.  

In [ ]:
# Obtener el número de entradas únicas en cada columna con datos categóricos
object_nunique = list(map(lambda col: X_train[col].nunique(), object_cols))
d = dict(zip(object_cols, object_nunique))

# Imprimir el número de entradas únicas por columna, en orden ascendente
sorted(d.items(), key=lambda x: x[1])

### **Paso 3:** Investigando la cardinalidad

### Parte A

La salida anterior muestra, para cada columna con datos categóricos, el número de valores únicos en esa columna. Por ejemplo, la columna `'Street'` en los datos de entrenamiento tiene dos valores únicos: `'Grvl'` y `'Pave'`, que corresponden a un camino de grava y a un camino asfaltado, respectivamente.

Nos referimos al número de entradas únicas de una variable categórica como la **cardinalidad** de esa variable categórica. Por ejemplo, la variable `'Street'` tiene una cardinalidad de 2.

Utiliza la salida anterior para responder a las preguntas que se presentan a continuación.

In [ ]:
# Rellena la línea de abajo: ¿Cuántas variables categóricas en los datos de entrenamiento tienen una cardinalidad mayor que 10?
high_cardinality_numcols = #____

# Rellena la línea de abajo: ¿Cuántas columnas son necesarias para codificar en one-hot la variable 'Vecindario' en los datos de entrenamiento?
num_cols_neighborhood = #____

### **Parte B**

Para conjuntos de datos grandes con muchas filas, la codificación `one-hot` puede ampliar considerablemente el tamaño del conjunto de datos. Por esta razón, normalmente solo codificamos en one-hot las columnas con una cardinalidad relativamente baja. Luego, las columnas de alta cardinalidad pueden ser eliminadas del conjunto de datos o podemos usar la codificación ordinal.

Como ejemplo, considera un conjunto de datos con 10.000 filas y que contiene una columna categórica con 100 entradas únicas.
- Si esta columna se reemplaza por su codificación one-hot correspondiente, ¿cuántas entradas se añaden al conjunto de datos?
- Si en su lugar se reemplaza por la codificación ordinal, ¿cuántas entradas se añaden?

Utiliza tus respuestas para completar las líneas a continuación.

In [ ]:
# Rellena la línea de abajo: ¿Cuántas entradas se añaden al conjunto de datos al
# reemplazar la columna con una codificación one-hot?
OH_entries_added = #____

# Rellena la línea de abajo: ¿Cuántas entradas se añaden al conjunto de datos al
# reemplazar la columna con una codificación ordinal?
label_entries_added = #____

A continuación, experimentarás con la codificación one-hot. Pero, en lugar de codificar todas las variables categóricas del conjunto de datos, solo crearás una codificación one-hot para las columnas con una cardinalidad inferior a 10.

Ejecuta la celda de código a continuación sin cambios para definir `low_cardinality_cols` como una lista de Python que contiene las columnas que se codificarán mediante one-hot. Asimismo, `high_cardinality_cols` contiene una lista de columnas categóricas que serán eliminadas del conjunto de datos.

In [ ]:
# Columnas que serán codificadas mediante one-hot
low_cardinality_cols = [col for col in object_cols if X_train[col].nunique() < 10]

# Columnas que serán eliminadas del conjunto de datos
high_cardinality_cols = list(set(object_cols)-set(low_cardinality_cols))

print('Columnas categóricas que serán codificadas mediante one-hot:', low_cardinality_cols)
print('\nColumnas categóricas que serán eliminadas del conjunto de datos:', high_cardinality_cols)

### **Paso 4:** Codificación one-hot

Utiliza la siguiente celda de código para codificar en one-hot los datos en `X_train` y `X_valid`. Asigna los DataFrames preprocesados a `OH_X_train` y `OH_X_valid`, respectivamente.
- La lista completa de columnas categóricas en el conjunto de datos se puede encontrar en la lista de Python `object_cols`.
- Solo debes codificar en one-hot las columnas categóricas en `low_cardinality_cols`. Todas las demás columnas categóricas deben eliminarse del conjunto de datos.

In [ ]:
from sklearn.preprocessing import OneHotEncoder

# ¡Utiliza tantas líneas de código como necesites!

# Aplicar codificador one-hot a cada columna con datos categóricos
OH_encoder = # OneHotEncoder(handle_unknown='ignore')
OH_cols_train = # pd.DataFrame(OH_encoder.fit_transform(X_train[low_cardinality_cols]).toarray())
OH_cols_valid = # pd.DataFrame(OH_encoder.transform(X_valid[low_cardinality_cols]).toarray())

# La codificación one-hot eliminó el índice; se restaura
OH_cols_train.index = # X_train.index
OH_cols_valid.index = # X_valid.index

# Eliminar columnas categóricas (serán reemplazadas por codificación one-hot)
num_X_train = # X_train.drop(object_cols, axis=1)
num_X_valid = # X_valid.drop(object_cols, axis=1)

# Añadir las columnas codificadas en one-hot a las características numéricas
OH_X_train = # pd.concat([num_X_train, OH_cols_train], axis=1)
OH_X_valid = # pd.concat([num_X_valid, OH_cols_valid], axis=1)

# Asegurar que todas las columnas sean de tipo cadena
OH_X_train.columns = # OH_X_train.columns.astype(str)
OH_X_valid.columns = # OH_X_valid.columns.astype(str)

Ejecuta la siguiente celda de código para obtener el MAE de este método.

In [ ]:
print("MAE from Approach 3 (One-Hot Encoding):")
print(score_dataset(OH_X_train, OH_X_valid, y_train, y_valid))

Ejecuta la siguiente celda de código, para recordar el rendimiento de los tre enfoques.

In [ ]:
print("MAE from Approach 1 (Drop categorical variables):")
print(score_dataset(drop_X_train, drop_X_valid, y_train, y_valid))

print("\nMAE from Approach 2 (Ordinal Encoding):")
print(score_dataset(label_X_train, label_X_valid, y_train, y_valid))

print("\nMAE from Approach 3 (One-Hot Encoding):")
print(score_dataset(OH_X_train, OH_X_valid, y_train, y_valid))

Si deseas seguir trabajando para mejorar tu rendimiento, puedes modificar tu código y repetir el proceso. Hay mucho margen de mejora a medida que avances y aprendas.

### **Genera predicciones de prueba.**

Después de completar el paso 4, si deseas usar lo aprendido para hacer predicciones con el enfoque que dió el mejor rendimiento `Ordinal Encoder` y guardar tus resultados, deberás preprocesar los datos de prueba `X_test` antes de generar las predicciones.